In [1]:
from promoter_protein_clip.model_transformer import SequenceCLIPTransformer
import torch
from huggingface_hub import hf_hub_download
import os
from promoter_protein_clip.utils import get_project_root
from promoter_protein_clip.evaluation import *
from sklearn.metrics import pairwise_distances
from transformers import GPT2LMHeadModel, AutoModel, AutoTokenizer
import pickle

/home/cameron/miniconda3/envs/clip_env_3/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### iModulonDB 0-shot coregulated gene retrieval evaluation

### Loading models

In [2]:
# Loading trained C3P models
repo_id = "dufaultc/contrastive-promoter-protein-pretraining"

filename = "C3P_100M.pth"
checkpoint_path = hf_hub_download(repo_id=repo_id, filename=filename )
device = torch.device("cuda:1" if torch.cuda.is_available() else "cpu")    
checkpoint= torch.load(checkpoint_path, map_location=device, weights_only=False)    
model_100 = SequenceCLIPTransformer(
    protein_model_name=checkpoint['args'].protein_model,
    promoter_embedding_dim=checkpoint['args'].promoter_embedding_dim,
    depth=checkpoint['args'].transformer_depth,
    num_heads=checkpoint['args'].transformer_num_heads,
    max_promoter_length=checkpoint['args'].max_promoter_length,
    projection_dim=checkpoint['args'].projection_dim,       
 ).to(device)
model_100 = torch.compile(model_100)
model_100.load_state_dict(checkpoint['model_state_dict'])  


filename = "C3P_25M.pth"
checkpoint_path = hf_hub_download(repo_id=repo_id, filename=filename )
device = torch.device("cuda:1" if torch.cuda.is_available() else "cpu")    
checkpoint= torch.load(checkpoint_path, map_location=device, weights_only=False)    
model_25 = SequenceCLIPTransformer(
    protein_model_name=checkpoint['args'].protein_model,
    promoter_embedding_dim=checkpoint['args'].promoter_embedding_dim,
    depth=checkpoint['args'].transformer_depth,
    num_heads=checkpoint['args'].transformer_num_heads,
    max_promoter_length=checkpoint['args'].max_promoter_length,
    projection_dim=checkpoint['args'].projection_dim,       
 ).to(device)
model_25 = torch.compile(model_25)
model_25.load_state_dict(checkpoint['model_state_dict'])  

filename = "C3P_5M.pth"
checkpoint_path = hf_hub_download(repo_id=repo_id, filename=filename )
device = torch.device("cuda:1" if torch.cuda.is_available() else "cpu")    
checkpoint= torch.load(checkpoint_path, map_location=device, weights_only=False)    
model_5 = SequenceCLIPTransformer(
    protein_model_name=checkpoint['args'].protein_model,
    promoter_embedding_dim=checkpoint['args'].promoter_embedding_dim,
    depth=checkpoint['args'].transformer_depth,
    num_heads=checkpoint['args'].transformer_num_heads,
    max_promoter_length=checkpoint['args'].max_promoter_length,
    projection_dim=checkpoint['args'].projection_dim,       
 ).to(device)
model_5 = torch.compile(model_5)
model_5.load_state_dict(checkpoint['model_state_dict']) 

filename = "C3P_1M.pth"
checkpoint_path = hf_hub_download(repo_id=repo_id, filename=filename )
device = torch.device("cuda:1" if torch.cuda.is_available() else "cpu")    
checkpoint= torch.load(checkpoint_path, map_location=device, weights_only=False)    
model_1 = SequenceCLIPTransformer(
    protein_model_name=checkpoint['args'].protein_model,
    promoter_embedding_dim=checkpoint['args'].promoter_embedding_dim,
    depth=checkpoint['args'].transformer_depth,
    num_heads=checkpoint['args'].transformer_num_heads,
    max_promoter_length=checkpoint['args'].max_promoter_length,
    projection_dim=checkpoint['args'].projection_dim,       
 ).to(device)
model_1 = torch.compile(model_1)
model_1.load_state_dict(checkpoint['model_state_dict'])  

Loading weights: 100%|██████████| 486/486 [00:00<00:00, 15821.17it/s]
[transformers] EsmModel LOAD REPORT from: facebook/esm2_t30_150M_UR50D
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Loading weights: 100%|██████████| 486/486 [00:00<00:00, 16829.57it/s]
[transformers] EsmModel LOAD REPORT from: facebook/esm2_t30_150M_UR50D
Key                       | Status     | 
--------------------------+------------+-
lm_head.

<All keys matched successfully>

In [3]:
#Loading PromoGen2 models
llm_model_base = GPT2LMHeadModel.from_pretrained("jinyuan22/promogen2-base").to(device)
llm_model_small = GPT2LMHeadModel.from_pretrained("jinyuan22/promogen2-small").to(device)
llm_model_xsmall = GPT2LMHeadModel.from_pretrained("jinyuan22/promogen2-xsmall").to(device)
llm_tokenizer = AutoTokenizer.from_pretrained("jinyuan22/promogen2-base")

llm_model_base = torch.compile(llm_model_base)
llm_model_base.eval()

llm_model_small = torch.compile(llm_model_small)
llm_model_small.eval()

llm_model_xsmall = torch.compile(llm_model_xsmall)
llm_model_xsmall.eval()

Loading weights: 100%|██████████| 76/76 [00:00<00:00, 14321.46it/s]


OptimizedModule(
  (_orig_mod): GPT2LMHeadModel(
    (transformer): GPT2Model(
      (wte): Embedding(10, 320)
      (wpe): Embedding(256, 320)
      (drop): Dropout(p=0.1, inplace=False)
      (h): ModuleList(
        (0-5): 6 x GPT2Block(
          (ln_1): LayerNorm((320,), eps=1e-05, elementwise_affine=True)
          (attn): GPT2Attention(
            (c_attn): Conv1D(nf=960, nx=320)
            (c_proj): Conv1D(nf=320, nx=320)
            (attn_dropout): Dropout(p=0.1, inplace=False)
            (resid_dropout): Dropout(p=0.1, inplace=False)
          )
          (ln_2): LayerNorm((320,), eps=1e-05, elementwise_affine=True)
          (mlp): GPT2MLP(
            (c_fc): Conv1D(nf=1280, nx=320)
            (c_proj): Conv1D(nf=320, nx=1280)
            (act): NewGELUActivation()
            (dropout): Dropout(p=0.1, inplace=False)
          )
        )
      )
      (ln_f): LayerNorm((320,), eps=1e-05, elementwise_affine=True)
    )
    (lm_head): Linear(in_features=320, out_features

In [4]:
#Loading gLM2 models
glm2_650_model = AutoModel.from_pretrained('tattabio/gLM2_650M', torch_dtype=torch.bfloat16, trust_remote_code=True).to(device)
glm2_150_model = AutoModel.from_pretrained('tattabio/gLM2_150M', torch_dtype=torch.bfloat16, trust_remote_code=True).to(device)
glm2_tokenizer = AutoTokenizer.from_pretrained('tattabio/gLM2_650M', trust_remote_code=True)

glm2_650_model = torch.compile(glm2_650_model)
glm2_650_model.eval()

glm2_150_model = torch.compile(glm2_150_model)
glm2_150_model.eval()

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 232/232 [00:00<00:00, 2460.99it/s]
[transformers] gLM2Model LOAD REPORT from: tattabio/gLM2_650M
Key                        | Status     |  | 
---------------------------+------------+--+-
lm_head.norm.weight        | UNEXPECTED |  | 
lm_head.proj_output.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 211/211 [00:00<00:00, 7060.89it/s]
[transformers] gLM2Model LOAD REPORT from: tattabio/gLM2_150M
Key                        | Status     |  | 
---------------------------+------------+--+-
lm_head.norm.weight        | UNEXPECTED |  | 
lm_head.proj_output.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


OptimizedModule(
  (_orig_mod): gLM2Model(
    (tok_embeddings): Embedding(37, 640)
    (encoder): TransformerLayers(
      (layers): ModuleList(
        (0-29): 30 x TransformerBlock(
          (attention): Attention(
            (wqkv): Linear(in_features=640, out_features=1920, bias=False)
            (wo): Linear(in_features=640, out_features=640, bias=False)
            (rotary_emb): RotaryEmbedding()
          )
          (feed_forward): FeedForward(
            (w1): Linear(in_features=640, out_features=1792, bias=False)
            (w2): Linear(in_features=1792, out_features=640, bias=False)
            (w3): Linear(in_features=640, out_features=1792, bias=False)
          )
          (attention_norm): RMSNorm()
          (ffn_norm): RMSNorm()
        )
      )
    )
  )
)

### Loading data, getting embeddings, and calculating NN

In [5]:
# Loading data for each iModulonDB database
eval_path = os.path.join(get_project_root(), "data/imodulondb")
samples = os.listdir(eval_path)

dfs = {}  
promoter_100_nearest_neighbours = {}  
promoter_25_nearest_neighbours = {}  
promoter_5_nearest_neighbours = {} 
promoter_1_nearest_neighbours = {}    
llm_base_promoter_nearest_neighbours = {}
llm_small_promoter_nearest_neighbours = {}
llm_xsmall_promoter_nearest_neighbours = {}
glm2_650_promoter_nearest_neighbours = {}
glm2_150_promoter_nearest_neighbours = {}
protein_nearest_neighbours = {}

for sample in tqdm.tqdm(samples):
    data_path = os.path.join(eval_path, sample)
    df = pd.read_csv(os.path.join(data_path, "data.csv"))
    df = df[df["operon_member"] == False]
    df=df.reset_index(drop=True)
    
    df = load_imodulon_data(df, os.path.join(data_path, "gene_presence_list.csv"))
    dfs[sample] = df

    promoter_100_embeddings = compute_promoter_embeddings(model_100,df,device,reverse_promoter=checkpoint['args'].reverse_promoter,max_promoter_length=checkpoint['args'].max_promoter_length, batch_size=256)        
    promoter_100_distances = pairwise_distances(promoter_100_embeddings, metric="cosine")
    promoter_100_nearest_neighbours[sample] = np.argsort(promoter_100_distances, axis=1) 
    
    promoter_25_embeddings = compute_promoter_embeddings(model_25,df,device,reverse_promoter=checkpoint['args'].reverse_promoter,max_promoter_length=checkpoint['args'].max_promoter_length, batch_size=256)        
    promoter_25_distances = pairwise_distances(promoter_25_embeddings, metric="cosine")
    promoter_25_nearest_neighbours[sample] = np.argsort(promoter_25_distances, axis=1)  
    
    promoter_5_embeddings = compute_promoter_embeddings(model_5,df,device,reverse_promoter=checkpoint['args'].reverse_promoter,max_promoter_length=checkpoint['args'].max_promoter_length, batch_size=256)        
    promoter_5_distances = pairwise_distances(promoter_5_embeddings, metric="cosine")
    promoter_5_nearest_neighbours[sample] = np.argsort(promoter_5_distances, axis=1)                                            
       
    promoter_1_embeddings = compute_promoter_embeddings(model_1,df,device,reverse_promoter=checkpoint['args'].reverse_promoter,max_promoter_length=checkpoint['args'].max_promoter_length, batch_size=256)        
    promoter_1_distances = pairwise_distances(promoter_1_embeddings, metric="cosine")
    promoter_1_nearest_neighbours[sample] = np.argsort(promoter_1_distances, axis=1)     
    
    promoter_strings = df["promoter"].to_list()
    promoter_strings = [promoter[-160:] if len(promoter) > 160 else promoter for promoter in promoter_strings]  
    embed_list = []
    for promoter in tqdm.tqdm(promoter_strings):
        yep = llm_tokenizer(f"<|bos|>5{promoter}3<|eos|>", return_tensors='pt')
        with torch.no_grad():
            embed_list.append(llm_model_base.transformer(yep.input_ids.to(device), output_hidden_states=True)['hidden_states'][-2][0].mean(dim=0).cpu().numpy())
    llm_promoter_embeddings= np.stack(embed_list, axis=0)        
    llm_promoter_distances = pairwise_distances(llm_promoter_embeddings, metric="cosine")
    llm_base_promoter_nearest_neighbours[sample] = np.argsort(llm_promoter_distances, axis=1) 

    promoter_strings = df["promoter"].to_list()
    promoter_strings = [promoter[-160:] if len(promoter) > 160 else promoter for promoter in promoter_strings]  
    embed_list = []
    for promoter in tqdm.tqdm(promoter_strings):
        yep = llm_tokenizer(f"<|bos|>5{promoter}3<|eos|>", return_tensors='pt')
        with torch.no_grad():
            embed_list.append(llm_model_small.transformer(yep.input_ids.to(device), output_hidden_states=True)['hidden_states'][-2][0].mean(dim=0).cpu().numpy())
    llm_promoter_embeddings= np.stack(embed_list, axis=0)        
    llm_promoter_distances = pairwise_distances(llm_promoter_embeddings, metric="cosine")
    llm_small_promoter_nearest_neighbours[sample] = np.argsort(llm_promoter_distances, axis=1)    

    promoter_strings = df["promoter"].to_list()
    promoter_strings = [promoter[-160:] if len(promoter) > 160 else promoter for promoter in promoter_strings]  
    embed_list = []
    for promoter in tqdm.tqdm(promoter_strings):
        yep = llm_tokenizer(f"<|bos|>5{promoter}3<|eos|>", return_tensors='pt')
        with torch.no_grad():
            embed_list.append(llm_model_xsmall.transformer(yep.input_ids.to(device), output_hidden_states=True)['hidden_states'][-2][0].mean(dim=0).cpu().numpy())
    llm_promoter_embeddings= np.stack(embed_list, axis=0)        
    llm_promoter_distances = pairwise_distances(llm_promoter_embeddings, metric="cosine")
    llm_xsmall_promoter_nearest_neighbours[sample] = np.argsort(llm_promoter_distances, axis=1)      
    
    promoter_strings = df["promoter"].to_list()
    promoter_strings = [promoter[-300:] if len(promoter) > 300 else promoter for promoter in promoter_strings]  
    embed_list = []
    for promoter in tqdm.tqdm(promoter_strings):
        yep = glm2_tokenizer(f"<+>{str.lower(promoter)}<+>", return_tensors='pt')
        with torch.no_grad():
            embed_list.append(glm2_650_model(yep.input_ids.to(device), output_hidden_states=True)['hidden_states'][-2][0].mean(dim=0).cpu())
    stacked = torch.stack(embed_list).float().numpy()
    glm2_promoter_embeddings= np.stack(stacked, axis=0)        
    glm2_promoter_distances = pairwise_distances(glm2_promoter_embeddings, metric="cosine")
    glm2_650_promoter_nearest_neighbours[sample]= np.argsort(glm2_promoter_distances, axis=1)         
    
    promoter_strings = df["promoter"].to_list()
    promoter_strings = [promoter[-300:] if len(promoter) > 300 else promoter for promoter in promoter_strings]  
    embed_list = []
    for promoter in tqdm.tqdm(promoter_strings):
        yep = glm2_tokenizer(f"<+>{str.lower(promoter)}<+>", return_tensors='pt')
        with torch.no_grad():
            embed_list.append(glm2_150_model(yep.input_ids.to(device), output_hidden_states=True)['hidden_states'][-2][0].mean(dim=0).cpu())
    stacked = torch.stack(embed_list).float().numpy()
    glm2_promoter_embeddings= np.stack(stacked, axis=0)        
    glm2_promoter_distances = pairwise_distances(glm2_promoter_embeddings, metric="cosine")
    glm2_150_promoter_nearest_neighbours[sample]= np.argsort(glm2_promoter_distances, axis=1)  
    
    protein_embeddings = compute_protein_embeddings(model_1, df, batch_size=256, device=device)
    protein_distances = pairwise_distances(protein_embeddings, metric="cosine")
    protein_nearest_neighbours[sample]=np.argsort(protein_distances, axis=1)          
        

100%|██████████| 2571/2571 [00:04<00:00, 536.43it/s]
[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!
W1020 14:45:17.368000 1946729 site-packages/torch/_dynamo/convert_frame.py:1743] [0/8] torch._dynamo hit config.recompile_limit (8)
W1020 14:45:17.368000 1946729 site-packages/torch/_dynamo/convert_frame.py:1743] [0/8]    function: 'forward' (/home/cameron/.cache/huggingface/modules/transformers_modules/tattabio/gLM2_650M/08754cba59a1f97d517f873fad6c672d2b1abdc7/modeling_glm2.py:394)
W1020 14:45:17.368000 1946729 site-packages/torch/_dynamo/convert_frame.py:1743] [0/8]    last reason: 0/7: input_ids.size()[1] <= self._modules['encoder']._modules['layers']._modules['0']._modules['attention']._modules['rotary_emb']._cos_cached.size()[0]  # cos, sin = cos[:seqlen], sin[:seqlen]  # .cache/huggingface/modules/transformers_modules/tattabio/gLM2_650M/08754cba59a1f97d517f873fad6c672d2b1abdc7/modeling_glm2.py:42 in apply_rotary_emb_torch (_subclasses/fake_impls.py:939 

### Loading pre-computed Evo1 and Evo2 nearest neighbours

In [6]:
#Loading Evo2 nn
filename = "evo2_7B_nn.pickle"
nn_path = hf_hub_download(repo_id=repo_id, filename=filename )
with open(nn_path, 'rb') as handle:
    evo2_nearest_neighbours = pickle.load(handle) 

In [7]:
#Loading Evo1 nn
filename = "evo1_7B_nn.pickle"
nn_path = hf_hub_download(repo_id=repo_id, filename=filename )
with open(nn_path, 'rb') as handle:
    evo1_nearest_neighbours = pickle.load(handle) 

### Performing Top-1 any match evaluation

In [22]:
# Top-1 ANY MATCH

source_names = [
    'C3P-large (100M)',    
    'C3P-base (25M)',
    'C3P-small (5M)',
    'C3P-tiny (1M)',
    'Evo2 (7B)',
    'Evo1 (7B)',
    'PromoGen2 (149M)',
    'PromoGen2 (33M)',
    'PromoGen2 (8M)',
    'gLM2 (650M)',
    'gLM2 (150M)',
    'ESM2',
]
family = [
    'C3P',    
    'C3P',
    'C3P',
    'C3P',
    'Evo2 (7B)',
    'Evo1 (7B)',
    'PromoGen2',
    'PromoGen2',
    'PromoGen2',
    'gLM2',
    'gLM2',
    'protein',
]

k = 1
full_set = False

results_100 = []
se_100 = []
for sample in samples:
    out = top_k_accuracy(
        dfs[sample],
        'imodulons',
        promoter_100_nearest_neighbours[sample],
        k=k,
        full_set=full_set,
    )
    results_100.append(out[0])
    se_100.append(out[1])

results_25 = []
se_25 = []
for sample in samples:
    out = top_k_accuracy(
        dfs[sample],
        'imodulons',
        promoter_25_nearest_neighbours[sample],
        k=k,
        full_set=full_set,
    )
    results_25.append(out[0])
    se_25.append(out[1])   


results_5 = []
se_5 = []
for sample in samples:
    out = top_k_accuracy(
        dfs[sample],
        'imodulons',
        promoter_5_nearest_neighbours[sample],
        k=k,
        full_set=full_set,
    )
    results_5.append(out[0])
    se_5.append(out[1]) 
    
results_1 = []
se_1 = []
for sample in samples:
    out = top_k_accuracy(
        dfs[sample],
        'imodulons',
        promoter_1_nearest_neighbours[sample],
        k=k,
        full_set=full_set,
    )
    results_1.append(out[0])
    se_1.append(out[1])       

results_llm_base = []
se_llm_base = []
for sample in samples:
    out = top_k_accuracy(
        dfs[sample],
        'imodulons',
        llm_base_promoter_nearest_neighbours[sample],
        k=k,
        full_set=full_set,
    )
    results_llm_base.append(out[0])
    se_llm_base.append(out[1])

results_llm_small = []
se_llm_small = []
for sample in samples:
    out = top_k_accuracy(
        dfs[sample],
        'imodulons',
        llm_small_promoter_nearest_neighbours[sample],
        k=k,
        full_set=full_set,
    )
    results_llm_small.append(out[0])
    se_llm_small.append(out[1])
    

results_llm_xsmall = []
se_llm_xsmall = []
for sample in samples:
    out = top_k_accuracy(
        dfs[sample],
        'imodulons',
        llm_xsmall_promoter_nearest_neighbours[sample],
        k=k,
        full_set=full_set,
    )
    results_llm_xsmall.append(out[0])
    se_llm_xsmall.append(out[1])                

results_evo = []
se_evo = []
for sample in samples:
    out = top_k_accuracy(
        dfs[sample],
        'imodulons',
        evo2_nearest_neighbours[sample],
        k=k,
        full_set=full_set,
    )
    results_evo.append(out[0])
    se_evo.append(out[1]) 

results_evo_one = []
se_evo_one = []
for sample in samples:
    out = top_k_accuracy(
        dfs[sample],
        'imodulons',
        evo1_nearest_neighbours[sample],
        k=k,
        full_set=full_set,
    )
    results_evo_one.append(out[0])
    se_evo_one.append(out[1])            

results_glm2_650 = []
se_glm2_650 = []
for sample in samples:
    out = top_k_accuracy(
        dfs[sample],
        'imodulons',
        glm2_650_promoter_nearest_neighbours[sample],
        k=k,
        full_set=full_set,
    )
    results_glm2_650.append(out[0])
    se_glm2_650.append(out[1])   



results_glm2_150 = []
se_glm2_150 = []
for sample in samples:
    out = top_k_accuracy(
        dfs[sample],
        'imodulons',
        glm2_150_promoter_nearest_neighbours[sample],
        k=k,
        full_set=full_set,
    )
    results_glm2_150.append(out[0])
    se_glm2_150.append(out[1])     
    
results_protein = []
se_protein= []
for sample in samples:
    out = top_k_accuracy(
        dfs[sample],
        'imodulons',
        protein_nearest_neighbours[sample],
        k=k,
        full_set=full_set,
    )
    results_protein.append(out[0])
    se_protein.append(out[1])          

all_results = [results_100, results_25, results_5,results_1, results_evo, results_evo_one, results_llm_base,results_llm_small, results_llm_xsmall ,results_glm2_650, results_glm2_150, results_protein]
all_means = [np.array(r).mean() for r in all_results]

all_ses = [se_100, se_25, se_5,se_1, se_evo, se_evo_one, se_llm_base, se_llm_small,se_llm_xsmall,se_glm2_650, se_glm2_150, se_protein]
all_sem = [(math.sqrt((np.array(se)**2).sum())/len(se))*2 for se in all_ses]

results_df_any = pd.DataFrame({"Method": source_names, "Mean Top-1": all_means, "2*SEM Top-1": all_sem, "Family": family})
full_results_df_any =  pd.DataFrame({"Dataset": samples})
for source,results,ses in zip(source_names, all_results,all_ses):
    full_results_df_any[source]= results   
    ses = [se*2 for se in ses]   
    full_results_df_any[f"{source}_se"]= ses            


In [23]:
print(results_df_any)

              Method  Mean Top-1  2*SEM Top-1     Family
0   C3P-large (100M)    0.391716     0.008359        C3P
1     C3P-base (25M)    0.377742     0.008234        C3P
2     C3P-small (5M)    0.333172     0.008071        C3P
3      C3P-tiny (1M)    0.269618     0.007607        C3P
4          Evo2 (7B)    0.123581     0.005958  Evo2 (7B)
5          Evo1 (7B)    0.077269     0.004923  Evo1 (7B)
6   PromoGen2 (149M)    0.102195     0.005637  PromoGen2
7    PromoGen2 (33M)    0.092297     0.005382  PromoGen2
8     PromoGen2 (8M)    0.088745     0.005266  PromoGen2
9        gLM2 (650M)    0.114818     0.005906       gLM2
10       gLM2 (150M)    0.091025     0.005240       gLM2
11              ESM2    0.218257     0.007213    protein


In [14]:
full_results_df_any[['Dataset','C3P-large (100M)', 'C3P-large (100M)_se', 'Evo2 (7B)', 'Evo2 (7B)_se']].sort_values('Dataset')

,Dataset,C3P-large (100M),C3P-large (100M)_se,Evo2 (7B),Evo2 (7B)_se
14,a_baumannii_precise139,0.407745,0.046908,0.154897,0.034536
2,b_diazoefficiens_modulome,0.489305,0.029847,0.197861,0.023787
5,b_subtilis_modulome,0.469194,0.034356,0.131517,0.023266
13,b_subtilis_nicolas,0.625199,0.038664,0.162679,0.029479
8,c_glutamicum_modulome263,0.158837,0.034577,0.123043,0.031074
16,e_coli_modulome,0.336859,0.026964,0.074858,0.015013
9,e_coli_precise1k,0.313099,0.026213,0.091853,0.016325
1,e_coli_precise278,0.357143,0.035181,0.071429,0.018909
19,e_coli_precise815,0.304202,0.026673,0.068067,0.014602
24,e_coli_precise_mg1655,0.399166,0.036527,0.083449,0.020628


### Performing Top-1 full match evaluation

In [19]:
# Top-1 Full match

source_names = [
    'C3P-large (100M)',    
    'C3P-base (25M)',
    'C3P-small (5M)',
    'C3P-tiny (1M)',
    'Evo2 (7B)',
    'Evo1 (7B)',
    'PromoGen2 (149M)',
    'PromoGen2 (33M)',
    'PromoGen2 (8M)',
    'gLM2 (650M)',
    'gLM2 (150M)',
    'ESM2',
]
family = [
    'C3P',    
    'C3P',
    'C3P',
    'C3P',
    'Evo2 (7B)',
    'Evo1 (7B)',
    'PromoGen2',
    'PromoGen2',
    'PromoGen2',
    'gLM2',
    'gLM2',
    'Protein',
]

k = 1
full_set = True
results_100 = []
se_100 = []
for sample in samples:
    out = top_k_accuracy(
        dfs[sample],
        'imodulons',
        promoter_100_nearest_neighbours[sample],
        k=k,
        full_set=full_set,
    )
    results_100.append(out[0])
    se_100.append(out[1])

results_25 = []
se_25 = []
for sample in samples:
    out = top_k_accuracy(
        dfs[sample],
        'imodulons',
        promoter_25_nearest_neighbours[sample],
        k=k,
        full_set=full_set,
    )
    results_25.append(out[0])
    se_25.append(out[1])   


results_5 = []
se_5 = []
for sample in samples:
    out = top_k_accuracy(
        dfs[sample],
        'imodulons',
        promoter_5_nearest_neighbours[sample],
        k=k,
        full_set=full_set,
    )
    results_5.append(out[0])
    se_5.append(out[1]) 
    
results_1 = []
se_1 = []
for sample in samples:
    out = top_k_accuracy(
        dfs[sample],
        'imodulons',
        promoter_1_nearest_neighbours[sample],
        k=k,
        full_set=full_set,
    )
    results_1.append(out[0])
    se_1.append(out[1])       

results_llm_base = []
se_llm_base = []
for sample in samples:
    out = top_k_accuracy(
        dfs[sample],
        'imodulons',
        llm_base_promoter_nearest_neighbours[sample],
        k=k,
        full_set=full_set,
    )
    results_llm_base.append(out[0])
    se_llm_base.append(out[1])

results_llm_small = []
se_llm_small = []
for sample in samples:
    out = top_k_accuracy(
        dfs[sample],
        'imodulons',
        llm_small_promoter_nearest_neighbours[sample],
        k=k,
        full_set=full_set,
    )
    results_llm_small.append(out[0])
    se_llm_small.append(out[1])
    

results_llm_xsmall = []
se_llm_xsmall = []
for sample in samples:
    out = top_k_accuracy(
        dfs[sample],
        'imodulons',
        llm_xsmall_promoter_nearest_neighbours[sample],
        k=k,
        full_set=full_set,
    )
    results_llm_xsmall.append(out[0])
    se_llm_xsmall.append(out[1])                

results_evo = []
se_evo = []
for sample in samples:
    out = top_k_accuracy(
        dfs[sample],
        'imodulons',
        evo2_nearest_neighbours[sample],
        k=k,
        full_set=full_set,
    )
    results_evo.append(out[0])
    se_evo.append(out[1]) 

results_evo_one = []
se_evo_one = []
for sample in samples:
    out = top_k_accuracy(
        dfs[sample],
        'imodulons',
        evo1_nearest_neighbours[sample],
        k=k,
        full_set=full_set,
    )
    results_evo_one.append(out[0])
    se_evo_one.append(out[1])            

results_glm2_650 = []
se_glm2_650 = []
for sample in samples:
    out = top_k_accuracy(
        dfs[sample],
        'imodulons',
        glm2_650_promoter_nearest_neighbours[sample],
        k=k,
        full_set=full_set,
    )
    results_glm2_650.append(out[0])
    se_glm2_650.append(out[1])   



results_glm2_150 = []
se_glm2_150 = []
for sample in samples:
    out = top_k_accuracy(
        dfs[sample],
        'imodulons',
        glm2_150_promoter_nearest_neighbours[sample],
        k=k,
        full_set=full_set,
    )
    results_glm2_150.append(out[0])
    se_glm2_150.append(out[1])   
    
results_protein = []
se_protein= []
for sample in samples:
    out = top_k_accuracy(
        dfs[sample],
        'imodulons',
        protein_nearest_neighbours[sample],
        k=k,
        full_set=full_set,
    )
    results_protein.append(out[0])
    se_protein.append(out[1])       

all_results = [results_100, results_25, results_5,results_1, results_evo, results_evo_one, results_llm_base,results_llm_small, results_llm_xsmall ,results_glm2_650, results_glm2_150, results_protein]
all_means = [np.array(r).mean() for r in all_results]

all_ses = [se_100, se_25, se_5,se_1, se_evo, se_evo_one, se_llm_base, se_llm_small,se_llm_xsmall,se_glm2_650, se_glm2_150, se_protein]
all_sem = [(math.sqrt((np.array(se)**2).sum())/len(se))*2 for se in all_ses]

results_df_all = pd.DataFrame({"Method": source_names, "Mean Top-1": all_means, "2*SEM Top-1": all_sem, "Family": family})
full_results_df_all =  pd.DataFrame({"Dataset": samples})
for source,results,ses in zip(source_names, all_results,all_ses):
    full_results_df_all[source]= results   
    ses = [se*2 for se in ses] 
    full_results_df_all[f"{source}_se"]= ses                       

In [20]:
print(results_df_all)

              Method  Mean Top-1  2*SEM Top-1     Family
0   C3P-large (100M)    0.236347     0.007330        C3P
1     C3P-base (25M)    0.227767     0.007208        C3P
2     C3P-small (5M)    0.195754     0.006868        C3P
3      C3P-tiny (1M)    0.156399     0.006312        C3P
4          Evo2 (7B)    0.066429     0.004618  Evo2 (7B)
5          Evo1 (7B)    0.034684     0.003482  Evo1 (7B)
6   PromoGen2 (149M)    0.050290     0.004166  PromoGen2
7    PromoGen2 (33M)    0.047607     0.004017  PromoGen2
8     PromoGen2 (8M)    0.044822     0.003930  PromoGen2
9        gLM2 (650M)    0.059379     0.004597       gLM2
10       gLM2 (150M)    0.043292     0.003824       gLM2
11              ESM2    0.125345     0.005877    Protein


In [18]:
full_results_df_all[['Dataset','C3P-large (100M)', 'C3P-large (100M)_se', 'Evo2 (7B)', 'Evo2 (7B)_se']].sort_values('Dataset')

,Dataset,C3P-large (100M),C3P-large (100M)_se,Evo2 (7B),Evo2 (7B)_se
14,a_baumannii_precise139,0.280182,0.042868,0.084282,0.026518
2,b_diazoefficiens_modulome,0.331551,0.028109,0.134581,0.020377
5,b_subtilis_modulome,0.220379,0.028536,0.068720,0.017416
13,b_subtilis_nicolas,0.457735,0.039793,0.118022,0.025770
8,c_glutamicum_modulome263,0.100671,0.028463,0.076063,0.025077
16,e_coli_modulome,0.142392,0.019936,0.034988,0.010483
9,e_coli_precise1k,0.107827,0.017531,0.022364,0.008358
1,e_coli_precise278,0.216981,0.030264,0.032345,0.012989
19,e_coli_precise815,0.107563,0.017963,0.021849,0.008476
24,e_coli_precise_mg1655,0.250348,0.032312,0.048679,0.016051
